# Load libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## Step 1: Load and Clean Data

Three sequential cleaning operations:

1. **Drop nulls** — rows with any missing value are removed (primarily affects amenity distance columns where geocoding returned no result: 157,821 → 114,147, dropped 43,674)
2. **Drop duplicates** — `1 ROOM` (41 rows) and `MULTI-GENERATION` (43 rows) excluded due to insufficient sample size (< 50 each, < 0.1% of data); outside the typical buyer use case (114,147 → 114,071, dropped 76)
3. **Drop rare flat types** — `1 ROOM` (41 rows) and `MULTI-GENERATION` (43 rows) excluded due to insufficient sample size (< 50 each, < 0.1% of data); outside the typical buyer use case (114,071 → 113,981, dropped 90)
4. **Drop 2020** — excluded due to the COVID-19 structural break; abnormal market conditions unrepresentative of typical pricing relationships (113,981 → 96,736, dropped 17,245)

In [2]:
df_raw = pd.read_csv("../../merged_data/hdb_with_amenities_macro.csv")
print(f"Initial shape: {df_raw.shape}")

df = df_raw.dropna()
print(f"After dropping nulls: {df.shape} (dropped {len(df_raw) - len(df)})")

# Remove duplicate rows to avoid repeated transactions affecting model training
n_before = len(df)
df = df.drop_duplicates()
print(f"After dropping duplicates: {df.shape} (dropped {n_before - len(df)})")

# Excluded due to insufficient sample size (fewer than 50 transactions each),
# representing less than 0.1% of the dataset and falling outside the typical buyer use case.
mask_flat = ~df["flat_type"].isin(["1 ROOM", "MULTI-GENERATION"])
n_before = len(df)
df = df[mask_flat]
print(f"After dropping 1 ROOM and MULTI-GENERATION: {df.shape} (dropped {n_before - len(df)})")

# 2020 is excluded due to the COVID-19 structural break which caused abnormal market conditions
# unrepresentative of typical pricing relationships.
df["year_temp"] = pd.to_datetime(df["month"]).dt.year
n_before = len(df)
df = df[df["year_temp"] != 2020].drop(columns="year_temp")
print(f"After dropping 2020: {df.shape} (dropped {n_before - len(df)})")

df['log_resale_price_real'] = np.log(df['resale_price_real'])  # Log-transform target — preserves resale_price_real for metric computation

# Count of primary schools within 1 km — derived from pipe-separated names already in the dataset.
# Captures school density independently of nearest-school distance.
df['num_primary_1km'] = df['primary_schools_1km'].apply(
    lambda x: len(x.split('|')) if pd.notna(x) and x != '' else 0
)
print(f"num_primary_1km — mean: {df['num_primary_1km'].mean():.2f}, max: {df['num_primary_1km'].max()}")

Initial shape: (157821, 37)
After dropping nulls: (114147, 37) (dropped 43674)
After dropping duplicates: (114071, 37) (dropped 76)
After dropping 1 ROOM and MULTI-GENERATION: (113981, 37) (dropped 90)
After dropping 2020: (96736, 37) (dropped 17245)
num_primary_1km — mean: 2.99, max: 9


# Step 2: Feature Engineering

## Step 2: Feature Engineering

Five new columns created from existing raw columns:

| New column | Source | Logic |
|-----------|--------|-------|
| `remaining_lease_years` | `remaining_lease` | Parse `"61 years 04 months"` → `61.33` (handles missing months) |
| `floor_category` | `storey_range` | Extract lower floor from `"10 TO 12"` → `10`; bucket as Low (1–5), Mid (6–12), High (13+) |
| `year` | `month` | Integer year — used for stratification only, **not** a model feature |
| `num_schools` | `primary_schools_1km` | Count number of primary schools within 1km |
| `num_parks` | `parks_1km` | Count number of parks within 1km |

**Target variable:** `log_resale_price_real` — using RPI-adjusted prices means the model captures structural flat-level pricing drivers rather than market-wide appreciation, since `year` is excluded as a feature.


In [3]:
import re

def parse_remaining_lease(s):
    match = re.match(r"(\d+) years?(?: (\d+) months?)?", str(s))
    if match:
        years = int(match.group(1))
        months = int(match.group(2)) if match.group(2) else 0
        return round(years + months / 12, 2)
    return np.nan

df["remaining_lease_years"] = df["remaining_lease"].apply(parse_remaining_lease)

# Extract lower floor number from storey_range (e.g., "10 TO 12" → 10)
df["floor_lower"] = df["storey_range"].str.extract(r"^(\d+)").astype(int)
df["floor_category"] = pd.cut(
    df["floor_lower"],
    bins=[0, 5, 12, float("inf")],
    labels=["Low", "Mid", "High"],
    right=True
)

# Year for stratification only — will NOT be used as a model feature
df["year"] = pd.to_datetime(df["month"]).dt.year


# Count number of primary schools within 1km
df["num_schools_1km"] = df["primary_schools_1km"].str.count(r"\|") + 1

# Count number of parks within 1km
df["num_parks_1km"] = df["parks_1km"].str.count(r"\|") + 1



# Target variable: log_resale_price_real
# RPI-adjusted prices are used so the model learns purely from flat characteristics
# rather than market-wide price appreciation over time, since year is not a model feature.
target = "log_resale_price_real"

print(df[["remaining_lease", "remaining_lease_years", "storey_range",
          "floor_lower", "floor_category", "year", 
          "primary_schools_1km", "num_schools_1km", "parks_1km", "num_parks_1km"]].head(10))


          remaining_lease  remaining_lease_years storey_range  floor_lower  \
23333   64 years 01 month                  64.08     01 TO 03            1   
23334   64 years 01 month                  64.08     07 TO 09            7   
23335            59 years                  59.00     04 TO 06            4   
23336  58 years 02 months                  58.17     04 TO 06            4   
23337   58 years 01 month                  58.08     01 TO 03            1   
23338  64 years 02 months                  64.17     07 TO 09            7   
23339  58 years 02 months                  58.17     04 TO 06            4   
23340   59 years 01 month                  59.08     04 TO 06            4   
23341            64 years                  64.00     07 TO 09            7   
23342  54 years 04 months                  54.33     04 TO 06            4   

      floor_category  year                                primary_schools_1km  \
23333            Low  2021  ANG MO KIO PRIMARY SCHOOL|MAYFLO

## Step 3: Stratified Train/Test Split (80/20)

**80/20 ratio:** With ~97,000 transactions, yields ~77,000 training and ~19,000 test rows. The training set is large enough for stable estimation; the test set is large enough for statistically reliable RMSE and linlin loss estimates. A 70/30 split is only preferred for datasets of a few thousand rows.

**Stratification key: `town + flat_type + year`** — three reasons:
1. **Town** — 26 HDB towns with substantially different price levels; random sampling could over-represent expensive towns in training.
2. **Flat type** — Highly imbalanced (4 ROOM ~42% vs 2 ROOM ~2%); stratifying prevents minority types being underrepresented.
3. **Year** — Despite RPI adjustment, residual structural differences exist across years (post-COVID demand surge, cooling measures); ensures proportional representation of every year in both sets.

In [4]:
from sklearn.model_selection import train_test_split

df["strat_key"] = (df["town"].astype(str) + "_" +
                   df["flat_type"].astype(str) + "_" +
                   df["year"].astype(str))

strat_counts = df["strat_key"].value_counts()
valid_keys = strat_counts[strat_counts >= 2].index
n_before = len(df)
df = df[df["strat_key"].isin(valid_keys)]
print(f"Dropped {n_before - len(df)} rows with singleton strat_key combinations. Remaining: {len(df):,}")

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["strat_key"])
print(f"Train size: {len(train_df):,} | Test size: {len(test_df):,}")

print("\nYear distribution (%):")
train_year = train_df["year"].value_counts(normalize=True).sort_index().rename("Train %")
test_year = test_df["year"].value_counts(normalize=True).sort_index().rename("Test %")
year_dist = pd.concat([train_year, test_year], axis=1)
print(year_dist.applymap(lambda x: f"{x:.2%}"))

print("\nFlat type distribution (%):")
train_flat = train_df["flat_type"].value_counts(normalize=True).rename("Train %")
test_flat = test_df["flat_type"].value_counts(normalize=True).rename("Test %")
flat_dist = pd.concat([train_flat, test_flat], axis=1)
print(flat_dist.applymap(lambda x: f"{x:.2%}"))

Dropped 7 rows with singleton strat_key combinations. Remaining: 96,729
Train size: 77,383 | Test size: 19,346

Year distribution (%):
     Train %  Test %
year                
2021  21.91%  21.91%
2022  19.93%  19.98%
2023  18.78%  18.77%
2024  20.62%  20.61%
2025  18.76%  18.73%

Flat type distribution (%):
          Train %  Test %
flat_type                
4 ROOM     41.87%  41.88%
3 ROOM     25.56%  25.54%
5 ROOM     23.60%  23.60%
EXECUTIVE   6.75%   6.77%
2 ROOM      2.22%   2.21%


## Step 4: Random Forest Model

**Feature matrix:**

| Type | Features | Treatment |
|------|----------|-----------|
| Continuous (9) | `remaining_lease_years`, `nearest_train_dist_m`, `num_primary_1km`, 6 amenity distances | Fit on train only |
| Categorical — flat type (4) | 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE | One-hot; reference = 2 ROOM |
| Categorical — town (25) | All towns except ANG MO KIO | One-hot; reference = ANG MO KIO |
| Categorical — floor (2) | Mid, High | One-hot; reference = Low (floors 1–5) |

**Excluded features:**
- `floor_area_sqm` — not a user-facing input at inference time; users select flat type but do not enter exact sqm. Flat type dummies implicitly capture size.

In [6]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import numpy as np

# =========================================
# Features 
# =========================================
continuous_features = [
    "remaining_lease_years", "nearest_train_dist_m",
    "dist_nearest_hawker_m", "dist_nearest_primary_m", "num_primary_1km",
    "dist_nearest_park_m", "dist_nearest_sportsg_m",
    "dist_nearest_mall_m", "dist_nearest_healthcare_m"  
]

categorical_features = ["flat_type", "town", "floor_category"]

# =========================================
# One-hot encoding 
# =========================================
train_encoded = pd.get_dummies(train_df, columns=categorical_features, drop_first=False)
test_encoded = pd.get_dummies(test_df, columns=categorical_features, drop_first=False)

# Identify dummy columns
dummy_cols = [c for c in train_encoded.columns
              if any(c.startswith(f"{f}_") for f in categorical_features)]

# =========================================
# Combine features 
# =========================================
X_train = pd.concat([train_encoded[continuous_features],
                     train_encoded[dummy_cols].astype(int)], axis=1)

X_test = pd.concat([test_encoded[continuous_features],
                    test_encoded[dummy_cols].astype(int)], axis=1)

# Align test columns to train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Targets
y_train = train_df[target]          
y_test = test_df[target]

y_train_actual = train_df['resale_price_real']
y_test_actual = test_df['resale_price_real']

# =========================================
# Train Random Forest
# =========================================
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# =========================================
# Predictions
# =========================================
y_test_pred_log = rf_model.predict(X_test)
y_test_pred = np.exp(y_test_pred_log)   # convert back to price

# =========================================
# Evaluation 
# =========================================
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def linlin_loss(y_true, y_pred, underpredict_weight=2.0):
    errors = np.array(y_true) - np.array(y_pred)
    loss = np.where(errors > 0, underpredict_weight * errors, -errors)
    return np.mean(loss)

test_rmse = rmse(y_test_actual, y_test_pred)
test_linlin = linlin_loss(y_test_actual, y_test_pred, underpredict_weight=2.0)

print("=== RANDOM FOREST MODEL EVALUATION ===")
print(f"RMSE:        ${test_rmse:,.0f}")
print(f"Linlin Loss: ${test_linlin:,.0f}")

=== RANDOM FOREST MODEL EVALUATION ===
RMSE:        $41,008
Linlin Loss: $42,330
